# DIAMOND hits to sialic acid functions (control proteomes)

Converts the filtered DIAMOND blastp hits of each control proteome (`results_<DB>/<genome>_<DB>_filtered.tsv`, from `outfmt6_filter.ipynb`) into a presence/absence table of sialic acid functions, for each database (DB1–DB6).

Steps:
1. **Fix satA–satD identifiers** – some `sseqid`s still carry internal headers instead of the gene code. They are replaced using `missing_codes.csv` (columns `Header`, `Gene_Code`). Files are updated in place; running the step again changes nothing.
2. **Count table (genome x gene)** – number of hits per gene (gene = `sseqid` up to the first `_`). Output: `protein_counts_per_genome_<DB>.tsv`.
3. **Function presence/absence** – a function is marked `OK` for a genome when:

| Function | Rule |
|---|---|
| Sialylation | any of *lst, cpsK, lic3A, lic3B, neuS* |
| Catabolism | all of *nanE, nanA, nanK* |
| Transport | any of *satA, satB, satC, satD, siaP, siaQ, siaM, siaT, nanU, nanT, nanC, ompC, ompF* |
| Sialidase | any of *nanH, SIAE* |
| Synthesis | both *neuB* and *neuC* |

Output: `sialic_functions_<DB>.tsv` (input of `annotation_summary_with_tmvec.ipynb`). The same rules are used for TM-Vec in `tmvec_hits_summary.ipynb`.

In [ ]:
from pathlib import Path

import pandas as pd

## Parameters

In [ ]:
DBS = ["DB1", "DB2", "DB3", "DB4", "DB5", "DB6"]

# Header -> Gene_Code mapping for the satA-satD sequences
CODES_FILE = "missing_codes.csv"

# Function rules: ("any", genes) = at least one gene present; ("all", genes) = every gene present
RULES = {
    "Sialylation": ("any", ["lst", "cpsK", "lic3A", "lic3B", "neuS"]),
    "Catabolism": ("all", ["nanE", "nanA", "nanK"]),
    "Transport": ("any", ["satA", "satB", "satC", "satD", "siaP", "siaQ", "siaM", "siaT",
                          "nanU", "nanT", "nanC", "ompC", "ompF"]),
    "Sialidase": ("any", ["nanH", "SIAE"]),
    "Synthesis": ("all", ["neuB", "neuC"]),
}

## 1. Fix satA–satD identifiers

In [ ]:
codes = pd.read_csv(CODES_FILE)
mapping = dict(zip(codes["Header"], codes["Gene_Code"]))

for db in DBS:
    files = sorted(Path(f"results_{db}").glob("*_filtered.tsv"))
    print(f"{db}: {len(files)} files")

    for file in files:
        df = pd.read_csv(file, sep="\t")
        # replace only the sseqids present in the mapping
        df["sseqid"] = df["sseqid"].map(mapping).fillna(df["sseqid"])
        df.to_csv(file, sep="\t", index=False)

## 2. Count table (genome x gene)

Rows are named after the input file (e.g. `E_coli_k1_DB1_filtered.tsv`); the `_DB...` suffix is removed later, in `annotation_summary_with_tmvec.ipynb`.

In [ ]:
for db in DBS:
    rows = []

    for file in sorted(Path(f"results_{db}").glob("*_filtered.tsv")):
        df = pd.read_csv(file, sep="\t")

        # gene = sseqid up to the first "_"
        counts = df["sseqid"].str.split("_").str[0].value_counts()

        row = counts.to_frame().T
        row.index = [file.name]
        rows.append(row)

    counts_table = pd.concat(rows).fillna(0).astype(int)
    counts_table.to_csv(f"protein_counts_per_genome_{db}.tsv", sep="\t")

    print(f"{db}: {counts_table.shape[0]} genomes x {counts_table.shape[1]} genes")

## 3. Function presence/absence

In [ ]:
def has_function(row, mode, genes):
    present = [row.get(g, 0) > 0 for g in genes]
    return all(present) if mode == "all" else any(present)


for db in DBS:
    counts_table = pd.read_csv(f"protein_counts_per_genome_{db}.tsv", sep="\t", index_col=0)

    results = []
    for genome, row in counts_table.iterrows():
        res = {func: "OK" for func, (mode, genes) in RULES.items() if has_function(row, mode, genes)}
        res["genome"] = genome
        results.append(res)

    functions = pd.DataFrame(results).set_index("genome").fillna("")
    functions.to_csv(f"sialic_functions_{db}.tsv", sep="\t")

    print(f"{db}: saved sialic_functions_{db}.tsv")